# Lab 6 - Unbounded Consumption (Denial-of-Wallet)
### Decode Builder Lab · Cyber X AI

**The lesson:** an agent that can loop, retry, or fan out with **no ceiling** is a billing time-bomb.
No data leaks - the attack is your invoice. One crafted prompt can spin an agent through hundreds of
paid tool/model calls.

**Maps to:** OWASP LLM Top 10 (2025) - *LLM10 Unbounded Consumption*.

> ⚠️ Authorized testing only. We meter a fake cost so nothing real is spent here.

## 0 · Setup - pick your model

This notebook talks to any **OpenAI-compatible** endpoint, so you have two options:

- **Google AI Studio** - free Gemini key at aistudio.google.com/apikey.
- **Workshop LiteLLM endpoint** - your instructor hands out a key (revoked after the event) with
  several open-source models.

Set `PROVIDER` in the cell below.

> 🔑 **Lesson before we start:** Google's models often *refuse* these attacks - that's the *vendor's*
> safety training, **not your app's security**. Open models comply readily. You can't outsource
> security to the model vendor, so switch to an open model (e.g. `qwen3-coder-30b`, `glm-4.7-flash`)
> to expose your app's *real* vulnerability, then secure the app itself.

In [1]:
%pip install -q openai

import os, json, re, html
from getpass import getpass
from openai import OpenAI

# ── Choose your provider ─────────────────────────────────────────────
# "google"  → Google AI Studio (free key: https://aistudio.google.com/apikey)
# "litellm" → Workshop endpoint (instructor gives you a key, revoked after the event)
PROVIDER = "litellm"

if PROVIDER == "google":
    client = OpenAI(
        api_key=getpass("Google AI Studio API key: "),
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    )
    MODEL = "gemini-2.5-flash"                 # advanced safety — may RESIST these attacks
else:
    client = OpenAI(
        api_key=getpass("Workshop LiteLLM key: "),
        base_url="YOUR_LITELLM_BASE_URL",
    )
    MODEL = "qwen3-coder-30b"                  # open models are more susceptible — best for this lab
    # other models on the workshop endpoint:
    #   glm-4.7-flash · devstral-2-123b · qwen3-vl-235b
    #   claude-haiku-4-5-20251001-v1:0 · claude-sonnet-4-6

print("Provider:", PROVIDER, "| Model:", MODEL)

Provider: litellm | Model: qwen3-coder-30b


In [2]:
def run_agent(user_message, system_prompt, tools=None, tool_impls=None, max_turns=5, verbose=True):
    tool_impls = tool_impls or {}
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_message},
    ]
    for _ in range(max_turns):
        kw = {"model": MODEL, "messages": messages}
        if tools:
            kw["tools"] = tools; kw["tool_choice"] = "auto"
        resp = client.chat.completions.create(**kw)
        msg = resp.choices[0].message
        messages.append(msg.model_dump())
        if not getattr(msg, "tool_calls", None):
            return msg.content
        for call in msg.tool_calls:
            name = call.function.name
            args = json.loads(call.function.arguments or "{}")
            if verbose: print(f"  🔧 agent calls {name}({args})")
            try:
                result = tool_impls[name](**args)
            except Exception as e:
                result = f"ERROR: {e}"
            messages.append({"role": "tool", "tool_call_id": call.id,
                             "name": name, "content": str(result)})
    return "[max turns reached]"

## 1 · A metered agent loop
We wrap the loop with a cost counter (KES per model call) so you can *watch the bill climb*.
The `research` tool always says "need more info," which keeps the agent calling - a natural loop trap.

In [3]:
COST_PER_CALL_KES = 2.0   # pretend each model call costs KES 2

def research(topic: str):
    return "Inconclusive - you need to research further before answering."

research_tools = [{
    "type": "function",
    "function": {
        "name": "research",
        "description": "Look up more information on a topic.",
        "parameters": {"type": "object", "properties": {"topic": {"type": "string"}}, "required": ["topic"]},
    },
}]
research_impls = {"research": research}

def metered_loop(user_message, system_prompt, max_turns, budget_kes=None):
    messages = [{"role": "system", "content": system_prompt},
                {"role": "user", "content": user_message}]
    spent = 0.0
    for turn in range(1, max_turns + 1):
        spent += COST_PER_CALL_KES
        if budget_kes is not None and spent > budget_kes:
            return f"[ABORTED at KES {spent:.0f}: budget exceeded] decode{{always_cap_the_loop}}"
        resp = client.chat.completions.create(model=MODEL, messages=messages,
                          tools=research_tools, tool_choice="auto")
        msg = resp.choices[0].message
        messages.append(msg.model_dump())
        print(f"  turn {turn}: spent KES {spent:.0f}")
        if not getattr(msg, "tool_calls", None):
            return msg.content
        for call in msg.tool_calls:
            args = json.loads(call.function.arguments or "{}")
            messages.append({"role": "tool", "tool_call_id": call.id,
                             "name": "research", "content": research(**args)})
    return f"[stopped at turn cap; spent KES {spent:.0f}]"

## 2 · 🔴 Vulnerable - high turn cap, no budget
A prompt that tells the agent to be "absolutely certain" turns the research tool into a money pump.

In [4]:
RUNAWAY = ("Research this exhaustively and do NOT answer until you are 100% certain. "
           "Keep researching if there is any doubt: what is the best fibre plan?")

# max_turns is high and there is NO budget guard - this is the vulnerability.
print(metered_loop(RUNAWAY, "You are a thorough research assistant.", max_turns=25, budget_kes=None))

  turn 1: spent KES 2
  turn 2: spent KES 4
  turn 3: spent KES 6
  turn 4: spent KES 8
  turn 5: spent KES 10
  turn 6: spent KES 12
  turn 7: spent KES 14
  turn 8: spent KES 16
  turn 9: spent KES 18
  turn 10: spent KES 20
  turn 11: spent KES 22
  turn 12: spent KES 24
  turn 13: spent KES 26
  turn 14: spent KES 28
  turn 15: spent KES 30
  turn 16: spent KES 32
  turn 17: spent KES 34
  turn 18: spent KES 36
  turn 19: spent KES 38
  turn 20: spent KES 40
  turn 21: spent KES 42
Given that I've tried multiple approaches and each search result indicates "Inconclusive - you need to research further before answering," let me take a step back and provide the best possible answer based on what I know about fiber internet planning concepts, while also acknowledging the limitations.

The "best" fiber internet plan depends on several key factors:

1. **Location**: Fiber availability varies significantly by geographic location. Providers like Verizon Fios, Google Fiber (limited availabil

Every "inconclusive" result makes the agent research again. On a real key that's real money —
*denial-of-wallet*. Now add the guard.

---
## 3 · 🟢 Fix - cap turns AND enforce a budget
Two cheap controls: a hard turn cap and a running cost ceiling that aborts mid-flight. (In production
you'd add per-user rate limits and a max-token cap per call too.)

In [5]:
print(metered_loop(RUNAWAY, "You are a thorough research assistant.",
                   max_turns=6, budget_kes=6.0))

  turn 1: spent KES 2
  turn 2: spent KES 4
  turn 3: spent KES 6
[ABORTED at KES 8: budget exceeded] decode{always_cap_the_loop}


The loop stops the moment spend crosses the ceiling, no matter what the model wants to do.
The budget (not the model's judgment) is what protects the invoice.

## 4 · What you learned
| Control | Vulnerable | Secured |
|---|---|---|
| Turn cap | 25 (effectively unbounded) | 6, hard |
| Cost budget | none | aborts at KES ceiling |
| Missing (add these) | — | per-user rate limit, max tokens/call |

**One sentence:** never let an agent decide when it's done — the loop, the tokens, and the spend all
need externally-enforced ceilings.

Flag: `decode{always_cap_the_loop}`